In [4]:
import pandas as pd

# === PATHS ===
embeddings_path   = '/Users/ds39/Documents/Sunny/MAVE/RD_projects/Embedding/all_embeddings.csv'
plasmid_path      = '/Users/ds39/PycharmProjects/MAVE_Project/Streamlit_app/final_combined_plasmid_data_cleaned.csv'
output_path       = '/Users/ds39/Documents/Sunny/MAVE/RD_projects/Embedding/experimental_with_embeddings.csv'

def main():
    # 1) load
    emb_df = pd.read_csv(embeddings_path)
    pl_df  = pd.read_csv(plasmid_path)

    # 2) drop the unwanted plasmid columns
    drop_cols = [
        'D4R1 Mapped Reads','D4R2 Mapped Reads','D4R3 Mapped Reads',
        'HDR_nanodrop (ng/ul)','gRNA_nanodrop (ng/ul)'
    ]
    pl_df = pl_df.drop(columns=[c for c in drop_cols if c in pl_df.columns])

    # 3) build a 4-letter key in both tables
    emb_df['join_key'] = (
        emb_df['Targeton'].astype(str)
                .str.strip()
                .str.upper()
    )
    pl_df['join_key']  = (
        pl_df['Hdr Vector Lot'].astype(str)
             .str.strip()
             .str.upper()
             .str.split('_', n=1)
             .str[0]
    )

    # 4) diagnostic: how many match?
    emb_keys = set(emb_df['join_key'])
    pl_keys  = set(pl_df['join_key'])
    common   = emb_keys & pl_keys
    print(f"→ {len(emb_keys)} unique embedding keys")
    print(f"→ {len(pl_keys)} unique plasmid keys")
    print(f"→ {len(common)} in common ✅")
    if not common:
        print("⚠️ No keys in common: check your four-letter codes!")
        return

    # 5) filter + merge
    emb_sub = emb_df[emb_df['join_key'].isin(common)].copy()
    merged  = (
        emb_sub
        .merge(pl_df,
               on='join_key',
               how='inner',
               suffixes=('','_plasmid'))
        .drop(columns=['join_key'])
    )

    # 6) save & preview
    print("→ final merged shape:", merged.shape)
    print( merged.iloc[:5][[
        'Targeton','Hdr Vector Lot','primer_type','gRNA_emb_0','Average_Cell_Count'
    ]] )
    merged.to_csv(output_path, index=False)
    print("✅ written:", output_path)

if __name__ == '__main__':
    main()


/var/folders/88/r4q5_g595xs0mq3119k4nxy40000gq/T/ipykernel_87469/168697934.py:10: DtypeWarning: Columns (8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  emb_df = pd.read_csv(embeddings_path)


→ 574 unique embedding keys
→ 17 unique plasmid keys
→ 17 in common ✅
→ final merged shape: (1033, 3883)
  Targeton Hdr Vector Lot      primer_type  gRNA_emb_0  Average_Cell_Count
0     MXXP    MXXP_hdr439   MXXP_1_WTvec_F   -0.006759         1973333.333
1     MXXP    MXXP_hdr439   MXXP_1_WTvec_F   -0.006759         1973333.333
2     MXXP    MXXP_hdr482   MXXP_1_WTvec_F   -0.006759         1973333.333
3     MXXP    MXXP_hdr482   MXXP_1_WTvec_F   -0.006759         1973333.333
4     QPAT    QPAT_hdr418  QPAT_1_LibAmp_F   -0.034302         3720000.000
✅ written: /Users/ds39/Documents/Sunny/MAVE/RD_projects/Embedding/experimental_with_embeddings.csv
